In [1]:
# Base Agent

In [2]:
from typing import Dict, Any, List, Union, Optional, Type, Callable, Literal
from pydantic import BaseModel, Field
import uuid
import os
import json
import inspect

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command, Send

from src.haive.core.engine.agent.agent import AgentArchitectureConfig, AgentArchitecture, register_agent
from src.haive.core.engine.aug_llm import AugLLMConfig, compose_runnable
from src.haive.core.models.llm.base import AzureLLMConfig
from src.haive.core.utils.visualize_graph_utils import render_and_display_graph
from src.config.settings import RESOURCES_DIR



class BaseAgentConfig(AgentArchitectureConfig):
    """
    Configuration for the simplest possible agent architecture.
    Wraps an AugLLM in a single-node graph.
    """
    engine: Union[AugLLMConfig, AgentArchitectureConfig] = Field(
        default_factory=lambda: AugLLMConfig(llm_config=AzureLLMConfig(model="gpt-4o")),
        description="LLM configuration for the BaseAgent."
    )
    input_mapping: Optional[Dict[str, str]] = Field(
        default=None,
        description="Maps state fields to LLM input fields."
    )
    output_mapping: Optional[Dict[str, str]] = Field(
        default=None,
        description="Maps LLM output fields to state fields."
    )
    node_name: str = Field(
        default="agent_node",
        description="Name for the single node in the graph."
    )
    should_setup_workflow: bool = Field(
        default=True,
        description="Whether to set up the workflow."
    )
    should_compile: bool = Field(
        default=True,
        description="Whether to compile the graph."
    )
    should_visualize_graph: bool = Field(
        default=False,
        description="Whether to visualize the graph."
    )
    visualize_graph_output_name: Optional[str] = Field(
        default=None,
        description="Output file for graph visualization."
    )
    
    def build_agent(self) -> "BaseAgent":
        """Build the BaseAgent from this configuration."""
        # Ensure state_schema is a Pydantic model
        if isinstance(self.state_schema, dict):
            #from state_schema_manager import StateSchemaManager
            schema_manager = StateSchemaManager(self.state_schema)
            self.state_schema = schema_manager.get_model()
        return BaseAgent(config=self)


# The registration will happen at the end of the file
class BaseAgent(AgentArchitecture):
    """
    The simplest possible agent architecture. Wraps an AugLLM in a single-node graph.
    """
    def __init__(self, config: BaseAgentConfig):
        self.input_mapping = config.input_mapping
        self.output_mapping = config.output_mapping
        self.node_name = config.node_name
        
        # Initialize the base agent
        super().__init__(config)
    
    def setup_workflow(self):
        """
        Set up a simple single-node workflow.
        
        The workflow consists of a single node that processes input through the AugLLM
        and routes directly to END.
        """
        # Create the node function with explicit END routing
        node_fn = create_node_function(
            config=self.config.engine if isinstance(self.config.engine, AugLLMConfig) else AugLLMConfig(),
            input_mapping=self.input_mapping,
            output_mapping=self.output_mapping,
            next_node=END  # Explicit END routing
        )
        
        # Add the node to the graph
        self.graph.add_node(self.node_name, node_fn)
        
        # Set the entry point
        self.graph.set_entry_point(self.node_name)
        self.graph.add_edge(self.node_name,END)

/home/will/Projects/haive/backend/haive/.venv/lib/python3.12/site-packages/pydantic/_internal/_config.py:345: UserWarning: Valid config keys have changed in V2:
* 'allow_population_by_field_name' has been renamed to 'populate_by_name'
* 'orm_mode' has been renamed to 'from_attributes'
  warnings.warn(message, UserWarning)
